# Hands-on Project: Spark Pipeline in Databricks (Solution)

This notebook will guide you through the process of building a Spark pipeline on Databricks using the Airbnb NYC listings dataset.

**About the data:** `AB_NYC_2019.csv` contains about 49,000 Airbnb listings in New York City in 2019. The data comes from [Inside Airbnb](https://insideairbnb.com/) and was published on Kaggle as [New York City Airbnb Open Data](https://www.kaggle.com/datasets/dgomonov/new-york-city-airbnb-open-data). Two columns are easy to confuse:
- `neighbourhood_group`: the borough (Manhattan, Brooklyn, Queens, Bronx, Staten Island)
- `neighbourhood`: the neighborhood inside the borough (Harlem, Midtown, Williamsburg, ...)

## Step 1: Initialize Spark Session
First, we need to get the Spark session.

In [ ]:
import os
from pyspark.sql import SparkSession

# On Databricks this returns the session that already exists.
# Locally (pip install pyspark) or on Colab it starts a new one.
spark = SparkSession.builder.getOrCreate()

# display(df) shows an interactive table only on Databricks: elsewhere we use show() instead
if "DATABRICKS_RUNTIME_VERSION" not in os.environ:
    def display(df):
        df.show(20, truncate=False)

## Step 2: Load the Dataset
We will load the dataset into a Spark DataFrame. We read the data directly from the GitHub repository of the course, so there is nothing to upload. pandas downloads the CSV and Spark turns it into a DataFrame. This is fine for small files; with real big data Spark would read the files directly from distributed storage (see [Lesson 7](https://github.com/Naviden/Introduction-to-Big-Data-and-NoSQL/blob/main/lessons/07_spark_basics_part1.md) for the other options).

In [ ]:
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/Naviden/Introduction-to-Big-Data-and-NoSQL/main/data"

def load_csv(path, **kwargs):
    """Read a CSV file of the course repository and return a Spark DataFrame."""
    pdf = pd.read_csv(f"{DATA_URL}/{path}", **kwargs)
    pdf = pdf.astype(object).where(pdf.notna(), None)  # pandas NaN -> Spark null
    return spark.createDataFrame(pdf)

In [ ]:
df = load_csv("airbnb/AB_NYC_2019.csv")

# Show the first few rows and the schema
df.show(5)
df.printSchema()

## Step 3: Selecting and Renaming Important Columns
To make the dataset more manageable, we'll select a subset of important columns and rename them for clarity.

In [ ]:
# Select specific columns and rename them for simplicity
selected_columns_df = df.select(
    df.id.alias("listing_id"),
    df.name.alias("listing_name"),
    df.host_id.alias("host_id"),
    df.neighbourhood_group.alias("borough"),
    df.neighbourhood.alias("neighborhood"),
    df.price.alias("price_per_night"),
    df.minimum_nights.alias("min_nights"),
    df.number_of_reviews.alias("reviews_count"),
    df.availability_365.alias("availability"),
    df.room_type.alias("room_type")
)

# Show the first few rows of the simplified DataFrame
selected_columns_df.show(5)

## Step 4: Data Exploration and Filtering
We'll filter the dataset to include listings in Manhattan with more than 5 reviews. Remember that Manhattan is a **borough**, not a neighborhood.

In [ ]:
# Filter listings in Manhattan with more than 5 reviews
filtered_df = selected_columns_df.filter(
    (selected_columns_df.borough == "Manhattan") & (selected_columns_df.reviews_count > 5)
)
print("Listings:", filtered_df.count())
filtered_df.show(5)

## Step 5: Data Transformation
Next, we'll create a new column that categorizes listings into price tiers: Low, Medium, and High.

In [ ]:
from pyspark.sql.functions import when

# Make sure 'price_per_night' is a numeric type
filtered_df = filtered_df.withColumn("price_per_night", filtered_df["price_per_night"].cast("float"))

# Create price tiers: Low (<$100), Medium ($100-$200), High (>$200)
transformed_df = filtered_df.withColumn(
    "price_tier",
    when(filtered_df.price_per_night < 100, "Low")
    .when((filtered_df.price_per_night >= 100) & (filtered_df.price_per_night <= 200), "Medium")
    .otherwise("High")
)
transformed_df.show(5)

## Step 6: Aggregation and Analysis
Let's calculate the average price per neighborhood in Manhattan and the average number of reviews for each price tier.

In [ ]:
# Make sure 'reviews_count' is a numeric type
transformed_df = transformed_df.withColumn("reviews_count", transformed_df["reviews_count"].cast("int"))

# Average price per neighborhood in Manhattan
avg_price_neighbourhood = transformed_df.groupBy("neighborhood").avg("price_per_night")
avg_price_neighbourhood.show()

# Average number of reviews per price tier
avg_reviews_price_tier = transformed_df.groupBy("price_tier").avg("reviews_count")
avg_reviews_price_tier.show()

## Final Thoughts
You can expand this project by exploring other datasets, adding new transformations, or experimenting with different types of aggregations. Some ideas:
- Which borough has the highest share of "Entire home/apt" listings?
- Are listings with a high `min_nights` cheaper per night?
- Sort the neighborhoods by average price, and keep only those with at least 50 listings.